## Load Datasets

In [1]:
from datasets import load_dataset, load_from_disk, Dataset, DatasetDict

In [2]:
CACHE_DIR = "../../hf_cache"

### Convert flores plus to opus format

In [ ]:
#DO NOT RUN AGAIN

si_ds = load_dataset("openlanguagedata/flores_plus", "sin_Sinh", cache_dir=CACHE_DIR)
en_ds = load_dataset("openlanguagedata/flores_plus", "eng_Latn", cache_dir=CACHE_DIR)
ta_ds = load_dataset("openlanguagedata/flores_plus", "tam_Taml", cache_dir=CACHE_DIR)

In [ ]:
#DO NOT RUN AGAIN

# Create test split from devtest
si_test = si_ds['devtest']
en_test = en_ds['devtest']
ta_test = ta_ds['devtest']
test_data = [
    {'translation': {'en': en_row['text'], 'si': si_row['text'], 'ta': ta_row['text']}}
    for si_row, en_row, ta_row in zip(si_test, en_test, ta_test)
]
test_dataset = Dataset.from_list(test_data)

# Create validation split from dev
si_val = si_ds['dev']
en_val = en_ds['dev']
ta_val = ta_ds['dev']
val_data = [
    {'translation': {'en': en_row['text'], 'si': si_row['text'], 'ta': ta_row['text']}}
    for si_row, en_row, ta_row in zip(si_val, en_val, ta_val)
]
val_dataset = Dataset.from_list(val_data)

# Combine into DatasetDict
dataset = DatasetDict({
    'devtest': test_dataset,
    'dev': val_dataset
})

#dataset.save_to_disk(CACHE_DIR + "/flores_combined")

In [3]:
ds_opus = load_dataset("Helsinki-NLP/opus-100", "en-si", cache_dir=CACHE_DIR)
ds_flores = load_from_disk(CACHE_DIR + "/flores_combined")

In [4]:
print(ds_opus)
print(ds_flores)
print(ds_opus['train'][0])
print(ds_flores['devtest'][0])

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 979109
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
DatasetDict({
    devtest: Dataset({
        features: ['translation'],
        num_rows: 1012
    })
    dev: Dataset({
        features: ['translation'],
        num_rows: 997
    })
})
{'translation': {'en': 'Boone!', 'si': 'බුන්!'}}
{'translation': {'en': '"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.', 'si': '"අපට දැන් මාස 4ක් වයසැති දියවැඩියාවෙන් පෙළුනු නමුත් දැන් දියවැඩියාව නොමැති මීයෙක් ඇත," ඔහු තවදුරටත් පැවසී ය.', 'ta': '"""எங்களிடம் இப்போது  4-மாத-வயதுடைய எலி ஒன்று உள்ளது, முன்னர் அதற்கு நீரிழிவு இருந்தது தற்போது இல்லை"" என்று அவர் மேலும் கூறினார்."'}}


### Load Tokenizers

In [5]:
import sys
import os
sys.path.insert(0, os.path.abspath("../../"))

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Split, ByteLevel, Whitespace, Metaspace
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC, Sequence
from custom_tokenizers.GPETokenizer import GPETokenizer

In [7]:
gpe_tokenizer_si = GPETokenizer(vocab_size=16000)
gpe_tokenizer_si.load("../../tokenizers_trained/gpe_si_1m_32k")

bpe_tokenizer_si = Tokenizer.from_file("../../tokenizers_trained/bpe_bytelevel_si_1m_32k/tokenizer.json")

superbpe_tokenizer_si_20K_extend_32K = Tokenizer.from_file("../../tokenizers_trained/superbpe_opus_100_si_10G_20K_extend_32K/tokenizer.json")

superbpe_tokenizer_si_28K_extend_32K = Tokenizer.from_file("../../tokenizers_trained/superbpe_opus_100_si_10G_28K_extend_32K/tokenizer.json")



In [8]:
import random

# Set random seed for reproducibility
random.seed(42)

# Pick 5 random samples from opus validation set
sample_indices = random.sample(range(len(ds_opus['validation'])), 5)
samples = [ds_opus['validation'][i] for i in sample_indices]

# Print encoded output from each tokenizer
for sample in samples:
    si_text = sample['translation']['si']
    
    print(f"\n")
    print(f"SI: {si_text}")

    ids = bpe_tokenizer_si.encode(si_text).ids
    tokens = [bpe_tokenizer_si.decode([id]) for id in ids]
    print(f"BPE:        {"|".join(tokens)}")
    
    ids = gpe_tokenizer_si.encode(si_text).ids
    tokens = [gpe_tokenizer_si.decode([id]) for id in ids]
    print(f"GPE:        {"|".join(tokens)}")
    
    ids = superbpe_tokenizer_si_20K_extend_32K.encode(si_text).ids
    tokens = [superbpe_tokenizer_si_20K_extend_32K.decode([id]) for id in ids]
    print(f"SuperBPE 20K Extended 32K: {"|".join(tokens)}")

    ids = superbpe_tokenizer_si_28K_extend_32K.encode(si_text).ids
    tokens = [superbpe_tokenizer_si_28K_extend_32K.decode([id]) for id in ids]
    print(f"SuperBPE 28K Extended 32K: {"|".join(tokens)}")





SI: ඒ කාරණාවෙන් මාව අර්ථවත් උනා , ඒකෙන් මට ශක්තිය ලැබුනා∙ .
BPE:         ඒ| ක|ා|රණ|ා|ව|ෙ|න|්| ම|ා|ව| අර|්|ථවත|්| උන|ා| ,| ඒක|ෙ|න|්| මට| ශක|්|ත|ි|ය| ල|ැ|බ|ු|න|ා∙| .
GPE:        ඒ| කාරණා|වෙන්| මාව| අර්ථ|වත්| උනා| ,| ඒකෙන්| මට| ශක්තිය| ලැබුනා|∙| .
SuperBPE 20K Extended 32K: ඒ| කාරණ|ාවෙන්| මාව| අර්ථ|වත්| උනා| ,| ඒකෙන් මට| ශක්තිය| ලැබුන|ා∙| .
SuperBPE 28K Extended 32K: ඒ| කාරණ|ාවෙන්| මාව| අර්ථ|වත්| උනා| ,| ඒකෙන්| මට| ශක්තිය| ලැබුන|ා∙| .


SI: එයාලා කොහේහරි වත්තක වැඩ කරනවා වෙනුවට, ඔයාගේ මිනිස්සුන්ට පුළුවන් එයාලගේ රටට සේවයක් කරන්න.
BPE:         එය|ා|ල|ා| ක|ො|හ|ේ|හර|ි| වත|්|තක| ව|ැ|ඩ| කරනව|ා| ව|ෙ|න|ු|වට|,| ඔය|ා|ග|ේ| ම|ි|න|ි|ස|්|ස|ු|න|්|ට| ප|ු|ළ|ු|වන|්| එය|ා|ලග|ේ| රටට| ස|ේ|වයක|්| කරන|්|න|.
GPE:        එයාලා| කොහේහරි| වත්ත|ක| වැඩ| කරනවා| වෙනුවට|,| ඔයාගේ| මිනිස්සුන්ට| පුළුවන්| එයාලගේ| රටට| සේවයක්| කරන්න|.
SuperBPE 20K Extended 32K: එයාලා| කොහේහරි| වත්ත|ක| වැඩ කරනවා| වෙනුවට|, ඔයාගේ| මිනිස්සුන්ට| පුළුවන්| එයාලගේ| රටට| සේවයක්| කරන්න.
SuperBPE 28K Extended 32K: එයාලා| කොහේහරි| වත්ත|ක| වැඩ කරනවා| වෙ

In [ ]:
def tokenizer_stats(tokenizer, sentences):
    total_tokens = 0
    total_words = 0
    total_chars = 0
    total_bytes = 0
    num_sentences = len(sentences)

    for s in sentences:
        # Tokenize
        tokens = tokenizer.encode(s, add_special_tokens=False).ids

        # Basic counts
        num_tokens = len(tokens)
        words = s.split()

        total_tokens += num_tokens
        total_words += len(words)
        total_chars += len(s)
        total_bytes += len(s.encode("utf-8"))

    # Avoid division by zero
    if total_tokens == 0 or total_words == 0 or num_sentences == 0:
        return {
            "avg_seq_len": 0,
            "fertility": 0,
            "char_per_token": 0,
            "byte_per_token": 0,
        }

    return {
        "avg_seq_len": total_tokens / num_sentences,
        "fertility": total_tokens / total_words,
        "char_per_token": total_chars / total_tokens,
        "byte_per_token": total_bytes / total_tokens,
    }

In [18]:
ds_opus_si = [ex["translation"]["si"] for ex in ds_opus["test"]]
ds_flores_si = [ex["translation"]["si"] for ex in ds_flores['devtest']]

In [20]:
bpe_stat_opus = tokenizer_stats(bpe_tokenizer_si, ds_opus_si)
gpe_stat_opus = tokenizer_stats(gpe_tokenizer_si, ds_opus_si)
superbpe_stat_opus_20K_extend_32K = tokenizer_stats(superbpe_tokenizer_si_20K_extend_32K, ds_opus_si)
superbpe_stat_opus_28K_extend_32K = tokenizer_stats(superbpe_tokenizer_si_28K_extend_32K, ds_opus_si)


In [23]:
bpe_stat_flores = tokenizer_stats(bpe_tokenizer_si, ds_flores_si)
gpe_stat_flores = tokenizer_stats(gpe_tokenizer_si, ds_flores_si)
superbpe_stat_flores_20K_extend_32K = tokenizer_stats(superbpe_tokenizer_si_20K_extend_32K, ds_flores_si)
superbpe_stat_flores_28K_extend_32K = tokenizer_stats(superbpe_tokenizer_si_28K_extend_32K, ds_flores_si)

In [22]:
import pandas as pd

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

# Collect results
results = {
    "BPE": bpe_stat_opus,
    "GPE": gpe_stat_opus,
    "SuperBPE 20K→32K": superbpe_stat_opus_20K_extend_32K,
    "SuperBPE 28K→32K": superbpe_stat_opus_28K_extend_32K,
}

# Convert to DataFrame
df = pd.DataFrame(results).T  # transpose for nice layout

# Optional: rename columns for clarity
df.columns = [
    "Avg Seq Len",
    "Fertility (tok/word)",
    "Chars per Token",
    "Bytes per Token"
]

# Round for readability
df = df.round(4)

print(df)

                  Avg Seq Len  Fertility (tok/word)  Chars per Token  Bytes per Token
BPE                   22.2445                3.6960           1.5668           4.0564
GPE                    8.5210                1.4158           4.0903          10.5894
SuperBPE 20K→32K       8.1200                1.3492           4.2923          11.1124
SuperBPE 28K→32K       8.2465                1.3702           4.2265          10.9419


In [24]:
# Collect results
results = {
    "BPE": bpe_stat_flores,
    "GPE": gpe_stat_flores,
    "SuperBPE 20K→32K": superbpe_stat_flores_20K_extend_32K,
    "SuperBPE 28K→32K": superbpe_stat_flores_28K_extend_32K,
}

# Convert to DataFrame
df = pd.DataFrame(results).T  # transpose for nice layout

# Optional: rename columns for clarity
df.columns = [
    "Avg Seq Len",
    "Fertility (tok/word)",
    "Chars per Token",
    "Bytes per Token"
]

# Round for readability
df = df.round(4)

print(df)

                  Avg Seq Len  Fertility (tok/word)  Chars per Token  Bytes per Token
BPE                   84.6660                4.1376           1.5298           4.0388
GPE                   31.0850                1.5191           4.1668          11.0006
SuperBPE 20K→32K      34.5524                1.6886           3.7487           9.8966
SuperBPE 28K→32K      34.2945                1.6760           3.7769           9.9711
